<a href="https://colab.research.google.com/github/marioluciofjr/codesoiram.python/blob/main/img_markdown.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Convertendo Imagens em Markdown

A intenção desse código é ser uuma adaptação do código do [professor Sandeco Macedo](https://youtu.be/hkGcnviwWAI?si=nsTm0E5UjnG_NAdH), a fim de converter imagens em markdown via Google Colab.

O código original do professor é este: https://github.com/sandeco/image_markdown

In [ ]:
# Instalando os pacotes necessários

!pip install -qU 'markitdown[all]'
!pip install -qU google-genai

print("Tudo Instalado")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.0/106.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/

## Como configurar a API KEY no Google Colab

Para utilizar sua chave API no Google Colab de forma segura:

1. Abra seu notebook no Google Colab
2. Na barra lateral esquerda, clique no ícone 🔑 (Secrets)
3. Clique em "+ Adicionar novo secret"
4. No campo "Nome", digite `senha`
5. No campo "Valor", cole sua chave API do Google AI Studio

> O código está configurado para acessar a chave por meio de `api_key = userdata.get('senha')`. Se preferir usar outro nome, modifique esta linha no código:

```python
# Usa a chave armazenada nos secrets do Colab sob o nome 'senha'
        api_key = userdata.get('senha')
  ```

In [ ]:
# Configurando a API GEMINI com o secret do Google Colab

from google.colab import userdata
import os
from google import genai

api_key = userdata.get("senha")
os.environ["GOOGLE_API_KEY"] = api_key

print("API Key configurada")

API Key configurada


In [ ]:
# Classe GeminiClientAdapter

import base64
import io
from PIL import Image

# Classes auxiliares para simular a resposta da OpenAI (sem alterações)
class _MockChoice:
    def __init__(self, content):
        self.message = type('obj', (object,), {'content': content})()

class _MockResponse:
    def __init__(self, content):
        self.choices = [_MockChoice(content)]

# A classe Adaptadora, agora responsável por criar seu próprio cliente
class GeminiClientAdapter:
    def __init__(self, api_key: str, model_name: str):
        """
        O construtor agora recebe a api_key e o model_name.
        Ele cria o cliente do genai internamente.
        """
        self.client = genai.Client(api_key=api_key)

        self.client_models = self.client.models
        self.model_name = model_name
        self.chat = self
        self.completions = self

    def create(self, model, messages, **kwargs):
        prompt = ""
        image_url = ""
        for part in messages[0]['content']:
            if part['type'] == 'text':
                prompt = part['text']
            elif part['type'] == 'image_url':
                image_url = part['image_url']['url']

        header, encoded = image_url.split(",", 1)
        image_data = base64.b64decode(encoded)
        image = Image.open(io.BytesIO(image_data))

        response = self.client_models.generate_content(
            model=f"models/{self.model_name}",
            contents=[prompt, image]
        )

        text_result = ""
        try:
            text_result = response.text
        except AttributeError:
            if response.candidates:
                text_result = response.candidates[0].content.parts[0].text

        return _MockResponse(content=text_result)

In [15]:
# Conversão de Imagem para Markdown

from markitdown import MarkItDown

model_name = "gemini-2.5-flash"
arquivo = input("COLOQUE O ENDEREÇO DA IMAGEM AQUI: ")

gemini_adapter_client = GeminiClientAdapter(api_key=api_key, model_name=model_name)

md = MarkItDown(llm_client=gemini_adapter_client,
                llm_model=model_name,
                llm_prompt="Em 3 parágrafos descreva a imagem detalhadamente em pt-br")

result = md.convert(arquivo)

#salve o resultado em "resultado.md"
with open("resultado.md", "w", encoding="utf-8") as f:
    f.write(result.text_content)

COLOQUE O ENDEREÇO DA IMAGEM AQUI: https://media.licdn.com/dms/image/v2/D4D22AQF8MXVJgst91A/feedshare-shrink_800/B4DZlLxp6VIYAg-/0/1757912932929?e=1761782400&v=beta&t=XMjIPeaXyZbRaaxPm5IluhTTya_OFrPzr6FRw1SDWlM
